In [21]:
!pip install -q spacy scikit-learn pandas PyPDF2 python-docx
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [22]:
import re
import spacy
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")
pd.set_option("display.max_colwidth", None)
print("Libraries loaded ✅")

Libraries loaded ✅


In [23]:
resumes = {
    "C001": """John Doe
    Experienced Python developer with 4 years in machine learning and NLP.
    Skills: Python, TensorFlow, Pandas, SQL, Scikit-learn, Git, Docker.
    Built resume parsing pipelines and worked with spaCy and NLTK.
    Education: B.Tech in Computer Science.""",

    "C002": """Priya Sharma
    Frontend developer skilled in React, JavaScript, HTML, CSS, and Redux.
    2 years experience building responsive web apps. Familiar with REST APIs
    and basic Node.js. Education: B.E. Information Technology.""",

    "C003": """Arun Kumar
    Data scientist with expertise in Python, Machine Learning, Deep Learning,
    TensorFlow, Keras, Pandas, NumPy, SQL and data visualization using
    Matplotlib and Seaborn. 3 years of experience in predictive modeling.""",

    "C004": """Sara Lee
    Java backend engineer with Spring Boot, Microservices, SQL, Docker,
    Kubernetes and AWS. 5 years experience designing scalable APIs.""",

    "C005": """Mohammed Ali
    Junior developer, fresher, knows Python, basic SQL, and Excel.
    Completed internship in data analysis using Pandas and Power BI.""",
}

job_descriptions = {
    "J001": """We are hiring a Machine Learning Engineer with strong Python skills,
    experience in TensorFlow or Keras, Pandas, Scikit-learn, SQL, and NLP
    libraries such as spaCy or NLTK. 2+ years experience required.""",

    "J002": """Looking for a Frontend Developer skilled in React, JavaScript,
    HTML, CSS and Redux, with experience consuming REST APIs.""",

    "J003": """Backend Engineer role requiring Java, Spring Boot, Microservices,
    Docker, Kubernetes, AWS and SQL. 3+ years experience.""",
}

print(f"{len(resumes)} resumes and {len(job_descriptions)} job descriptions loaded ✅")

5 resumes and 3 job descriptions loaded ✅


In [24]:
SKILL_DB = [
    "python", "java", "javascript", "react", "redux", "node.js", "html", "css",
    "sql", "nosql", "mongodb", "docker", "kubernetes", "aws", "azure", "gcp",
    "tensorflow", "keras", "pytorch", "scikit-learn", "pandas", "numpy",
    "matplotlib", "seaborn", "power bi", "excel", "spring boot", "microservices",
    "machine learning", "deep learning", "nlp", "spacy", "nltk", "git",
    "rest api", "data visualization", "predictive modeling",
]

def extract_skills(text, skill_db=SKILL_DB):
    """Extract known skills from resume text using normalized keyword matching."""
    text_lower = text.lower()
    text_norm = re.sub(r'[\s/]+', ' ', text_lower)
    found = set()
    for skill in skill_db:
        pattern = r'(?<!\w)' + re.escape(skill).replace(r'\-', r'[- ]') + r'(?!\w)'
        if re.search(pattern, text_norm):
            found.add(skill)
    return sorted(found)

resume_skills = {cid: extract_skills(text) for cid, text in resumes.items()}
for cid, skills in resume_skills.items():
    print(f"{cid}: {skills}")

C001: ['docker', 'git', 'machine learning', 'nlp', 'nltk', 'pandas', 'python', 'scikit-learn', 'spacy', 'sql', 'tensorflow']
C002: ['css', 'html', 'javascript', 'node.js', 'react', 'redux']
C003: ['data visualization', 'deep learning', 'keras', 'machine learning', 'matplotlib', 'numpy', 'pandas', 'predictive modeling', 'python', 'seaborn', 'sql', 'tensorflow']
C004: ['aws', 'docker', 'java', 'kubernetes', 'microservices', 'spring boot', 'sql']
C005: ['excel', 'pandas', 'power bi', 'python', 'sql']


In [25]:
def rank_resumes(job_text, resumes_dict):
    """Rank all resumes against a single job description by textual similarity."""
    candidate_ids = list(resumes_dict.keys())
    corpus = [job_text] + [resumes_dict[cid] for cid in candidate_ids]

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(corpus)

    job_vec = tfidf_matrix[0:1]
    resume_vecs = tfidf_matrix[1:]
    scores = cosine_similarity(job_vec, resume_vecs).flatten()

    ranking = pd.DataFrame({
        "candidate_id": candidate_ids,
        "match_score": (scores * 100).round(2),
        "skills": [", ".join(resume_skills[cid]) for cid in candidate_ids],
    }).sort_values("match_score", ascending=False).reset_index(drop=True)

    return ranking

ranking_J001 = rank_resumes(job_descriptions["J001"], resumes)
ranking_J001

,candidate_id,match_score,skills
0,C001,41.39,"docker, git, machine learning, nlp, nltk, pandas, python, scikit-learn, spacy, sql, tensorflow"
1,C003,27.93,"data visualization, deep learning, keras, machine learning, matplotlib, numpy, pandas, predictive modeling, python, seaborn, sql, tensorflow"
2,C004,13.91,"aws, docker, java, kubernetes, microservices, spring boot, sql"
3,C005,6.93,"excel, pandas, power bi, python, sql"
4,C002,5.60,"css, html, javascript, node.js, react, redux"


In [26]:
def filter_candidates(ranking_df, required_skills=None, min_score=30):
    """Keep only candidates who meet a minimum score AND have all required skills."""
    filtered = ranking_df[ranking_df["match_score"] >= min_score].copy()

    if required_skills:
        required_skills = [s.lower() for s in required_skills]
        def has_all(skills_str):
            candidate_skills = [s.strip() for s in skills_str.split(",")]
            return all(rs in candidate_skills for rs in required_skills)
        filtered = filtered[filtered["skills"].apply(has_all)]

    return filtered.reset_index(drop=True)

shortlist = filter_candidates(ranking_J001, required_skills=["python", "sql"], min_score=30)
print("Shortlisted candidates for J001:")
shortlist

Shortlisted candidates for J001:


,candidate_id,match_score,skills
0,C001,41.39,"docker, git, machine learning, nlp, nltk, pandas, python, scikit-learn, spacy, sql, tensorflow"


In [27]:
def match_candidates_to_jobs(resumes_dict, jobs_dict):
    """For every candidate, find the job description they match best."""
    results = []
    for cid, resume_text in resumes_dict.items():
        best_job, best_score = None, -1
        for jid, job_text in jobs_dict.items():
            corpus = [job_text, resume_text]
            tfidf = TfidfVectorizer(stop_words="english").fit_transform(corpus)
            score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0] * 100
            if score > best_score:
                best_job, best_score = jid, score
        results.append({
            "candidate_id": cid,
            "best_matching_job": best_job,
            "match_score": round(best_score, 2),
            "skills": ", ".join(resume_skills[cid]),
        })
    return pd.DataFrame(results).sort_values("match_score", ascending=False).reset_index(drop=True)

job_match_table = match_candidates_to_jobs(resumes, job_descriptions)
job_match_table

,candidate_id,best_matching_job,match_score,skills
0,C004,J003,64.24,"aws, docker, java, kubernetes, microservices, spring boot, sql"
1,C002,J002,45.76,"css, html, javascript, node.js, react, redux"
2,C001,J001,36.21,"docker, git, machine learning, nlp, nltk, pandas, python, scikit-learn, spacy, sql, tensorflow"
3,C003,J001,28.68,"data visualization, deep learning, keras, machine learning, matplotlib, numpy, pandas, predictive modeling, python, seaborn, sql, tensorflow"
4,C005,J001,8.06,"excel, pandas, power bi, python, sql"


In [28]:
# Create dictionaries
resumes = {}
resume_skills = {}

# Skill extraction function
def extract_skills(text):
    skills_list = [
        "python", "java", "c", "c++", "sql", "html", "css", "javascript",
        "react", "node", "mongodb", "machine learning", "deep learning",
        "tensorflow", "keras", "pandas", "numpy", "flask", "django",
        "power bi", "excel", "aws", "git"
    ]

    text = text.lower()
    found = [skill for skill in skills_list if skill in text]
    return found

In [29]:
from google.colab import files
import PyPDF2, io

def extract_text_from_pdf(file_bytes):
    reader = PyPDF2.PdfReader(io.BytesIO(file_bytes))
    return " ".join(page.extract_text() or "" for page in reader.pages)

uploaded = files.upload()  # select one or more .pdf or .txt resumes

for fname, content in uploaded.items():
    cid = fname.rsplit(".", 1)[0].upper().replace(" ", "_")
    if fname.lower().endswith(".pdf"):
        text = extract_text_from_pdf(content)
    else:
        text = content.decode("utf-8", errors="ignore")
    resumes[cid] = text
    resume_skills[cid] = extract_skills(text)
    print(f"Added {cid} ({len(text)} chars) — skills: {resume_skills[cid]}")

print("Re-run Cells 5-7 to include the new resumes in ranking / filtering / matching.")

Saving HARSHA_E_Resume (1).pdf to HARSHA_E_Resume (1) (2).pdf
Added HARSHA_E_RESUME_(1)_(2) (2638 chars) — skills: ['python', 'java', 'c', 'sql', 'html', 'css', 'javascript', 'node', 'django', 'excel', 'aws', 'git']
Re-run Cells 5-7 to include the new resumes in ranking / filtering / matching.


In [30]:
print("job_match_table" in globals())

True


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def match_candidates_to_jobs(resumes_dict, jobs_dict):
    """For every candidate, find the best matching job."""
    results = []

    for cid, resume_text in resumes_dict.items():
        best_job = None
        best_score = -1

        for jid, job_text in jobs_dict.items():
            corpus = [job_text, resume_text]

            tfidf = TfidfVectorizer(stop_words="english").fit_transform(corpus)
            score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0] * 100

            if score > best_score:
                best_score = score
                best_job = jid

        results.append({
            "candidate_id": cid,
            "best_matching_job": best_job,
            "match_score": round(best_score, 2),
            "skills": ", ".join(resume_skills[cid]),
        })

    return pd.DataFrame(results).sort_values(
        "match_score", ascending=False
    ).reset_index(drop=True)

In [35]:
summary = job_match_table.copy()
summary = summary[["candidate_id", "best_matching_job", "match_score", "skills"]]
summary.style.background_gradient(subset=["match_score"], cmap="Greens")

,candidate_id,best_matching_job,match_score,skills
0,HARSHA_E_RESUME_(1)_(2),J001,5.490000,"python, java, c, sql, html, css, javascript, node, django, excel, aws, git"
